In [1]:
import os, re, json, math, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

SEED = 42
set_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE)

DATA_DIR = "data/processed"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")

OUT_DIR = "runs/q2c_two_stage"
os.makedirs(OUT_DIR, exist_ok=True)


DEVICE: cuda


In [ ]:
# ============================================================
# load and clean
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

LABEL_COL  = "label"
SOURCE_COL = "source_text"
REPLY_COL  = "reply_text"

_url_re = re.compile(r"https?://\S+|www\.\S+")
_user_re = re.compile(r"@\w+")
_space_re = re.compile(r"\s+")

def clean_tweet(s):
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = s.replace("\n", " ").replace("\r", " ")
    s = _url_re.sub(" HTTPURL ", s)
    s = _user_re.sub(" @USER ", s)
    s = _space_re.sub(" ", s).strip()
    return s

LABEL_CANON = {
    "c": "comment", "comment": "comment",
    "s": "support", "support": "support",
    "d": "deny",    "deny": "deny",
    "q": "query",   "query": "query",
}

def canon_label(x):
    x = str(x).strip().lower()
    return LABEL_CANON.get(x, x)

for df in [train_df, dev_df, test_df]:
    df[SOURCE_COL] = df[SOURCE_COL].apply(clean_tweet)
    df[REPLY_COL]  = df[REPLY_COL].apply(clean_tweet)
    df[LABEL_COL]  = df[LABEL_COL].apply(canon_label)

print("Train label counts:\n", train_df[LABEL_COL].value_counts())
print("Unique labels (train):", sorted(train_df[LABEL_COL].unique().tolist()))


Train label counts:
 label
comment    2291
support     743
deny        273
query       262
Name: count, dtype: int64
Unique labels (train): ['comment', 'deny', 'query', 'support']


In [ ]:
MODEL_NAME = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

MAX_LEN = 192

def encode_pair(source: str, reply: str):
    hint = "TASK: Determine if the reply expresses stance toward the source."
    src = f"SOURCE: {source}"
    rep = f"REPLY: {reply}"
    return tokenizer(
        hint + " " + src,
        rep,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        return_tensors="pt",
    )


class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, label_map: dict):
        self.df = df.reset_index(drop=True)
        self.label_map = label_map

    def __len__(self): return len(self.df)

    def __getitem__(self, idx: int):
        r = self.df.iloc[idx]
        enc = encode_pair(r[SOURCE_COL], r[REPLY_COL])
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.label_map[r[LABEL_COL]], dtype=torch.long)
        return item


In [4]:
def plot_confusion(cm: np.ndarray, labels, title, save_path):
    fig = plt.figure(figsize=(6,5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45, ha="right")
    plt.yticks(ticks, labels)

    thresh = cm.max() * 0.6 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            v = cm[i, j]
            plt.text(j, i, str(v), ha="center", va="center",
                     color=("white" if v > thresh else "black"))

    plt.ylabel("True"); plt.xlabel("Pred")
    plt.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)

def macro_f1_from_logits(logits, labels):
    preds = logits.argmax(axis=1)
    return f1_score(labels, preds, average="macro")


In [ ]:

import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler

def make_class_weights(
    df: pd.DataFrame,
    label_map: dict,
    power: float = 0.8
) -> torch.Tensor:
    counts = df[LABEL_COL].value_counts().to_dict()

    w = np.zeros(len(label_map), dtype=np.float32)
    for lab, idx in label_map.items():
        w[idx] = 1.0 / max(1.0, float(counts.get(lab, 1.0)))

    w = w / w.mean()
    w = np.power(w, power)

    return torch.tensor(w, dtype=torch.float32)


def make_sampler(
    df: pd.DataFrame,
    label_map: dict
) -> WeightedRandomSampler:

    n = int(len(df))
    if n == 0:
        raise ValueError(
            "make_sampler(): received empty dataframe; cannot make sampler."
        )

    mapped = df[LABEL_COL].map(label_map)

    bad = df.loc[mapped.isna(), LABEL_COL].unique().tolist()
    if len(bad) > 0:
        raise ValueError(
            f"make_sampler(): found labels not in label_map: {bad}\n"
            f"Expected one of: {list(label_map.keys())}"
        )

    y = mapped.astype(int).values
    class_counts = np.bincount(y, minlength=len(label_map)).astype(np.float32)

    class_w = 1.0 / np.clip(class_counts, 1.0, None)
    sample_w = class_w[y]

    return WeightedRandomSampler(
        weights=torch.tensor(sample_w, dtype=torch.float32),
        num_samples=int(len(sample_w)),
        replacement=True
    )


import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedFocalLoss(nn.Module):

    def __init__(
        self,
        class_weights: torch.Tensor,
        gamma: float = 1.5,
        label_smoothing: float = 0.0
    ):
        super().__init__()
        self.register_buffer("w", class_weights)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        num_classes = logits.size(-1)

        if self.label_smoothing > 0.0:`
            with torch.no_grad():
                true_dist = torch.zeros_like(logits)
                true_dist.fill_(self.label_smoothing / (num_classes - 1))
                true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.label_smoothing)

            log_probs = F.log_softmax(logits, dim=-1)
            probs = log_probs.exp()

            pt = (probs * true_dist).sum(dim=1).clamp_min(1e-9)
            focal = (1.0 - pt) ** self.gamma

            w_exp = (self.w.unsqueeze(0) * true_dist).sum(dim=1)
            loss = -(w_exp * focal * (log_probs * true_dist).sum(dim=1))
            return loss.mean()

        
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()

        pt = probs.gather(1, target.unsqueeze(1)).squeeze(1).clamp_min(1e-9)
        focal = (1.0 - pt) ** self.gamma

        w = self.w.gather(0, target)
        loss = -(w * focal * log_probs.gather(1, target.unsqueeze(1)).squeeze(1))
        return loss.mean()



import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedCELoss(nn.Module):

    def __init__(self, class_weights: torch.Tensor, label_smoothing: float = 0.0):
        super().__init__()
        self.register_buffer("w", class_weights)
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return F.cross_entropy(
            logits,
            target,
            weight=self.w,
            label_smoothing=float(self.label_smoothing),
        )


In [ ]:
import inspect

def make_training_args_compat(**kwargs) -> TrainingArguments:

    sig = inspect.signature(TrainingArguments.__init__)
    valid = set(sig.parameters.keys())

    
    if "evaluation_strategy" in kwargs and "eval_strategy" in valid:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    if "save_strategy" in kwargs and "save_strategy" not in valid and "save_strategy" in kwargs:
        pass


    filtered = {k: v for k, v in kwargs.items() if k in valid}
    return TrainingArguments(**filtered)


In [7]:
class LossTrainer(Trainer):
    def __init__(self, loss_fn: nn.Module, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = loss_fn

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
# ============================================================
# Stage 1 (Comment vs Non-Comment)
# ============================================================

S1_LABEL_MAP = {"comment": 0, "noncomment": 1}
S1_ID2LABEL   = {0: "comment", 1: "noncomment"}

def make_s1_df(df):
    out = df.copy()
    out[LABEL_COL] = out[LABEL_COL].apply(lambda x: "comment" if x == "comment" else "noncomment")
    return out

train_s1 = make_s1_df(train_df)
dev_s1   = make_s1_df(dev_df)

print("[Stage1] train counts:\n", train_s1[LABEL_COL].value_counts())
print("[Stage1] dev counts:\n", dev_s1[LABEL_COL].value_counts())

ds_train_s1 = PairDataset(train_s1, S1_LABEL_MAP)
ds_dev_s1   = PairDataset(dev_s1, S1_LABEL_MAP)

w_s1 = make_class_weights(train_s1, S1_LABEL_MAP, power=0.5).to(DEVICE)
loss_s1 = WeightedCELoss(w_s1, label_smoothing=0.02)

model_s1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=S1_ID2LABEL, label2id=S1_LABEL_MAP
).to(DEVICE)

args_s1 = make_training_args_compat(
    output_dir=os.path.join(OUT_DIR, "stage1"),
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=6,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

def s1_compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"macro_f1": macro_f1_from_logits(logits, labels)}

trainer_s1 = LossTrainer(
    loss_fn=loss_s1,
    model=model_s1,
    args=args_s1,
    train_dataset=ds_train_s1,
    eval_dataset=ds_dev_s1,
    tokenizer=tokenizer,
    compute_metrics=s1_compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("\n=== TRAIN STAGE 1 ===")
trainer_s1.train()

print("\n=== DEV STAGE 1 ===")
s1_dev_out = trainer_s1.predict(ds_dev_s1)
print(s1_dev_out.metrics)

s1_dev_pred = s1_dev_out.predictions.argmax(axis=1)
s1_dev_true = s1_dev_out.label_ids
cm1 = confusion_matrix(s1_dev_true, s1_dev_pred, labels=[0,1])
plot_confusion(cm1, ["comment","noncomment"], "Stage1 Confusion (DEV)", os.path.join(OUT_DIR, "cm_stage1_dev.png"))
print("Saved:", os.path.join(OUT_DIR, "cm_stage1_dev.png"))




[Stage1] train counts:
 label
comment       2291
noncomment    1278
Name: count, dtype: int64
[Stage1] dev counts:
 label
comment       616
noncomment    334
Name: count, dtype: int64


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_6927/2969241869.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `LossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)



=== TRAIN STAGE 1 ===


Step,Training Loss,Validation Loss,Macro F1
200,0.618900,0.642611,0.661915
400,0.570200,0.663082,0.675406
600,0.478600,0.705519,0.650406



=== DEV STAGE 1 ===


{'test_loss': 0.6426105499267578, 'test_macro_f1': 0.6619152046783625, 'test_runtime': 1.4685, 'test_samples_per_second': 646.935, 'test_steps_per_second': 20.43}
Saved: runs/q2c_two_stage/cm_stage1_dev.png


In [ ]:


@torch.no_grad()
def stage1_logits_and_stats(df):
    tmp = make_s1_df(df)
    ds = PairDataset(tmp, S1_LABEL_MAP)
    out = trainer_s1.predict(ds)
    logits = out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    p_nc = probs[:, 1]
    pred = logits.argmax(axis=1)

    print("Stage1 p(noncomment) stats:",
          "min", float(p_nc.min()),
          "mean", float(p_nc.mean()),
          "max", float(p_nc.max()))
    print("Stage1 predicted noncomment rate:", float((pred == 1).mean()))
    return logits, p_nc, pred

_ = stage1_logits_and_stats(dev_df)


Stage1 p(noncomment) stats: min 0.1965947300195694 mean 0.47223806381225586 max 0.90183424949646
Stage1 predicted noncomment rate: 0.3684210526315789


In [ ]:


import numpy as np
import torch

@torch.no_grad()
def stage1_probs(df):
    """Return p(noncomment) for each example using Stage 1."""
    tmp = make_s1_df(df)
    ds = PairDataset(tmp, S1_LABEL_MAP)
    out = trainer_s1.predict(ds)
    logits = out.predictions  # (n,2)
    probs = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()
    return probs[:, 1]

def print_prob_stats(name, p):
    print(f"{name} p(noncomment) stats:",
          "min", float(np.min(p)),
          "mean", float(np.mean(p)),
          "max", float(np.max(p)))

p_nc_dev  = stage1_probs(dev_df)
p_nc_test = stage1_probs(test_df)
print_prob_stats("DEV", p_nc_dev)
print_prob_stats("TEST", p_nc_test)



DEV p(noncomment) stats: min 0.1965947300195694 mean 0.47223806381225586 max 0.90183424949646
TEST p(noncomment) stats: min 0.3785165250301361 mean 0.3785165250301361 max 0.3785165250301361


In [ ]:


S2_LABEL_MAP = {"support": 0, "deny": 1, "query": 2}
S2_ID2LABEL  = {0: "support", 1: "deny", 2: "query"}

def canon_label_stage2(x):
    x = str(x).strip()
    m = {
        "S":"support","D":"deny","Q":"query","C":"comment",
        "s":"support","d":"deny","q":"query","c":"comment",
        "support":"support","deny":"deny","query":"query","comment":"comment",
        "Support":"support","Deny":"deny","Query":"query","Comment":"comment",
    }
    return m.get(x, x.lower())

for df in [train_df, dev_df, test_df]:
    df[LABEL_COL] = df[LABEL_COL].apply(canon_label_stage2)

def filter_sdq(df: pd.DataFrame) -> pd.DataFrame:
    return df[df[LABEL_COL].isin(["support","deny","query"])].copy().reset_index(drop=True)

train_s2 = filter_sdq(train_df)
dev_s2   = filter_sdq(dev_df)

print("\n[Stage2] train_s2 size:", len(train_s2), "dev_s2 size:", len(dev_s2))
print("[Stage2] train_s2 label counts:\n", train_s2[LABEL_COL].value_counts())

def encode_pair(source: str, reply: str):
    src = f"SOURCE: {source}"
    rep = f"REPLY: {reply}"
    hint = "TASK: Decide stance of REPLY toward SOURCE as Support, Deny, or Query."
    return tokenizer(
        hint + " " + src,
        rep,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        return_tensors="pt",
    )


ds_train_s2 = PairDataset(train_s2, S2_LABEL_MAP)
ds_dev_s2   = PairDataset(dev_s2, S2_LABEL_MAP)


w_s2 = make_class_weights(train_s2, S2_LABEL_MAP, power=0.7).to(DEVICE)
loss_s2 = WeightedCELoss(w_s2, label_smoothing=0.05)

sampler_s2 = make_sampler(train_s2, S2_LABEL_MAP)

model_s2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=S2_ID2LABEL, label2id=S2_LABEL_MAP
).to(DEVICE)

args_s2 = make_training_args_compat(
    output_dir=os.path.join(OUT_DIR, "stage2"),
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1.5e-5,
    weight_decay=0.01,
    num_train_epochs=8,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

def s2_compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"macro_f1": macro_f1_from_logits(logits, labels)}

trainer_s2 = LossTrainer(
    loss_fn=loss_s2,
    model=model_s2,
    args=args_s2,
    train_dataset=ds_train_s2,
    eval_dataset=ds_dev_s2,
    tokenizer=tokenizer,
    compute_metrics=s2_compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer_s2.get_train_dataloader = lambda: DataLoader(
    trainer_s2.train_dataset,
    batch_size=trainer_s2.args.per_device_train_batch_size,
    sampler=sampler_s2,
)

print("\n=== TRAIN STAGE 2 ===")
trainer_s2.train()

print("\n=== DEV STAGE 2 ===")
s2_dev_out = trainer_s2.predict(ds_dev_s2)
print(s2_dev_out.metrics)

s2_dev_pred = s2_dev_out.predictions.argmax(axis=1)
s2_dev_true = s2_dev_out.label_ids
cm2 = confusion_matrix(s2_dev_true, s2_dev_pred, labels=[0,1,2])
plot_confusion(cm2, ["support","deny","query"], "Stage2 Confusion (DEV)", os.path.join(OUT_DIR, "cm_stage2_dev.png"))
print("Saved:", os.path.join(OUT_DIR, "cm_stage2_dev.png"))



[Stage2] train_s2 size: 1278 dev_s2 size: 334
[Stage2] train_s2 label counts:
 label
support    743
deny       273
query      262
Name: count, dtype: int64


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_6927/2969241869.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `LossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)



=== TRAIN STAGE 2 ===


Step,Training Loss,Validation Loss,Macro F1
200,0.719000,0.895937,0.636857
400,0.525500,0.936445,0.649313
600,0.384600,1.005363,0.665773



=== DEV STAGE 2 ===


{'test_loss': 0.895937442779541, 'test_macro_f1': 0.6368571482616426, 'test_runtime': 0.5502, 'test_samples_per_second': 607.019, 'test_steps_per_second': 19.992}
Saved: runs/q2c_two_stage/cm_stage2_dev.png


In [ ]:
# ============================================================
#  Combine stages (4-way)
# ============================================================

import os
import numpy as np
import torch
from sklearn.metrics import f1_score, classification_report

FOUR = ["support", "deny", "query", "comment"]
L4 = {lab: i for i, lab in enumerate(FOUR)}
I4 = {i: lab for lab, i in L4.items()}

def ensure_out_dir():
    global OUT_DIR
    if "OUT_DIR" not in globals():
        OUT_DIR = "runs/q2c_two_stage"
    os.makedirs(OUT_DIR, exist_ok=True)

@torch.no_grad()
def stage1_p_noncomment(df):
    tmp = make_s1_df(df)
    ds = PairDataset(tmp, S1_LABEL_MAP)
    out = trainer_s1.predict(ds)
    logits = out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()
    return probs[:, 1]

@torch.no_grad()
def stage2_argmax_labels(df_noncomment):
    ds = PairDataset(df_noncomment, S2_LABEL_MAP)
    out = trainer_s2.predict(ds)
    pred_ids = out.predictions.argmax(axis=1)
    inv = {v: k for k, v in S2_LABEL_MAP.items()}
    return [inv[int(i)] for i in pred_ids]

def combine_two_stage(df, p_thr=0.5):
    y_true = df[LABEL_COL].map(L4).values

    p_nc = stage1_p_noncomment(df)
    routed = (p_nc >= float(p_thr))
    routed_rate = float(np.mean(routed))

    y_pred = np.full(len(df), L4["comment"], dtype=int)

    if routed.any():
        df_r = df.loc[routed].copy()
        df_r[LABEL_COL] = "support"

        sdq_preds = stage2_argmax_labels(df_r)
        y_pred[routed] = [L4[x] for x in sdq_preds]

    return y_true, y_pred, routed_rate

def eval_4way(name, df, p_thr):
    y_true, y_pred, routed_rate = combine_two_stage(df, p_thr=p_thr)
    macro = float(f1_score(y_true, y_pred, average="macro", labels=list(range(4))))
    print(f"\n[{name}] p_thr={p_thr:.3f} | routed={routed_rate*100:.2f}% | macro-F1={macro:.4f}")
    print(classification_report(y_true, y_pred, target_names=FOUR, zero_division=0))
    return macro, routed_rate


p_grid = np.linspace(0.05, 0.95, 19)
best_thr, best_f1 = 0.5, -1.0

for t in p_grid:
    m, r = combine_two_stage(dev_df, p_thr=t)[0:3:2]
    y_true, y_pred, routed_rate = combine_two_stage(dev_df, p_thr=t)
    macro = float(f1_score(y_true, y_pred, average="macro", labels=list(range(4))))
    if routed_rate < 0.05:
        continue
    if macro > best_f1:
        best_f1, best_thr = macro, float(t)

print("\nBEST DEV (macro tuned) p_thr:", best_thr, "macro-F1:", best_f1)


_ = eval_4way("DEV final", dev_df, p_thr=best_thr)


_ = eval_4way("TEST final", test_df, p_thr=best_thr)


ensure_out_dir()

def save_preds(df, split_name, p_thr):
    y_true, y_pred, routed_rate = combine_two_stage(df, p_thr=p_thr)
    out = df.copy()
    out["gold"] = [I4[int(i)] for i in y_true]
    out["pred_final"] = [I4[int(i)] for i in y_pred]
    out["correct"] = (out["gold"] == out["pred_final"]).astype(int)
    out_path = os.path.join(OUT_DIR, f"preds_{split_name}.csv")
    out.to_csv(out_path, index=False)
    print("Saved:", out_path, "| routed_rate:", routed_rate)

save_preds(dev_df, "dev", best_thr)
save_preds(test_df, "test", best_thr)

print("\nFINAL CHOSEN p_thr:", best_thr)



BEST DEV (macro tuned) p_thr: 0.5499999999999999 macro-F1: 0.5152423646672848



[DEV final] p_thr=0.550 | routed=27.37% | macro-F1=0.5152
              precision    recall  f1-score   support

     support       0.53      0.50      0.52       167
        deny       0.41      0.15      0.22        71
       query       0.61      0.47      0.53        96
     comment       0.75      0.84      0.79       616

    accuracy                           0.69       950
   macro avg       0.57      0.49      0.52       950
weighted avg       0.67      0.69      0.67       950




[TEST final] p_thr=0.550 | routed=0.00% | macro-F1=0.2129
              precision    recall  f1-score   support

     support       0.00      0.00      0.00        94
        deny       0.00      0.00      0.00        71
       query       0.00      0.00      0.00       106
     comment       0.74      1.00      0.85       778

    accuracy                           0.74      1049
   macro avg       0.19      0.25      0.21      1049
weighted avg       0.55      0.74      0.63      1049



Saved: runs/q2c_two_stage/preds_dev.csv | routed_rate: 0.2736842105263158


Saved: runs/q2c_two_stage/preds_test.csv | routed_rate: 0.0

FINAL CHOSEN p_thr: 0.5499999999999999
